<div dir="rtl">

# 🏗️ حلول YOLO للواقع الحقيقي: حوّل الرؤية إلى بيانات ذكية! 📊

مرحباً بك في الجزء الثاني من رحلتنا! اليوم سننتقل من مجرد "الرؤية" إلى "التحليل واتخاذ القرار".

## **Ultralytics Solutions** 
 لتحويل الكشوفات الخام إلى رؤى عملية تفيد الأعمال.

## 🎯 ماذا سنتعلم اليوم؟

- تطبيق عدّ الكائنات في مناطق محددة باستخدام المضلعات.
- قياس السرعة والمسافة بين الأجسام التي يتم تتبعها.
- إنشاء رسوم بيانية وخرائط حرارية (Heatmaps) من الفيديوهات.
- تطبيق تغبيش الأجسام لحماية الخصوصية.
- دمج عدة حلول في نظام واحد متكامل.

---

## 📹 مصادر الفيديو المستخدمة

سنستخدم فيديوهات عينة من الملفات المحلية ومن GitHub:

| المصدر | الملف | حالة الاستخدام |
| :--- | :--- | :--- |
| **محلي** | `Pull_ups.mp4` | مراقبة التمارين الرياضية |
| **محلي** | `parking_slots.mp4` | إدارة مواقف السيارات |
| **محلي** | `Cars.mp4` | عينة فيديو مروري |

</div>


<div dir="rtl">

---

## 🛠️ 1. تجهيز بيئة العمل

**لماذا؟** لضمان وجود كافة المكتبات البرمجية والفيديوهات التي سنعمل عليها.

</div>


In [ ]:
# 🛠️ الخطوة 1: تثبيت المكتبات اللازمة وتجهيز الأدوات
import sys
# تثبيت مكتبات معالجة الصور والذكاء الاصطناعي
%pip install -q ultralytics opencv-python pandas matplotlib requests sahi

import cv2              # OpenCV لمعالجة الفيديو
import pandas as pd     # Pandas لتحليل البيانات
import matplotlib.pyplot as plt # Matplotlib للرسم البياني
import os                # OS للتعامل مع الملفات
import requests          # Requests لتحميل الملفات
from ultralytics import YOLO, solutions # YOLO للذكاء الاصطناعي
from IPython.display import Video      # لعرض الفيديو

# 📂 الخطوة 2: تحميل فيديوهات التجارب
videos = ["Pull_ups.mp4", "parking_slots.mp4", "Cars.mp4"]
for video_file in videos:
    if not os.path.exists(video_file):
        url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/{video_file}"
        print(f"⏳ جاري تحميل {video_file}...")
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            with open(video_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✓ تم تحميل: {video_file} بنجاح!")
        else:
            print(f"✗ فشل تحميل {video_file}.")


<div dir="rtl">

---

# المحطة الأولى: تحليل المناطق والحدود 📍

في هذه المحطة، سنركز على "أين" يقع الكائن وهل يعبر خطوطاً معينة.

</div>


<div dir="rtl">

## 📍 تمرين 1.1: عدّ الكائنات في مناطق مخصصة

### 💡 الفكرة التعليمية:
بدلاً من عدّ كل شيء في الصورة، سنقوم برسم "خط وهمي" أو "منطقة". أي سيارة تعبر هذا الخط سيتم عدّها.

**لماذا نختار هذه الخطوة؟**
لأننا في الواقع نهتم بالتدفق (Flow)، مثلاً: كم سيارة دخلت المواقف؟ وكم سيارة خرجت؟

**المهمة:** إجراء تحليل مقارن بين تقنيتين لتعريف مناطق العدّ 

1. **المنطقة الخطية (Line Region):** تعتمد على رصد عبور نقطة المركز للكائن لخط مستقيم. تتميز بالبساطة البرمجية ولكنها قد تتأثر بالتداخل في الزوايا الحادة.
2. **المنطقة المضلعة (Polygon Region):** توفر إطاراً مكانياً دقيقاً (Spatial Constraints) لضمان عدّ الكائنات التي تقع داخل حيّز جغرافي محدد فقط.

**الهدف:** تقييم تأثير هندسة المنطقة  على دقة البيانات المستخلصة، وتحديد النوع الأمثل للحد من القراءات الخاطئة الناتجة عن تداخل المسارات.

**الفيديو:** تقاطع مروري في المدينة.

</div>


In [ ]:
video_path = "Cars.mp4"

# تحميل الملف إذا لم يكن موجوداً
if not os.path.exists(video_path):
    url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/{video_path}"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

# Display the video
Video(video_path, width=640, height=360, embed=True)


<div dir="rtl">

# 🚗 تمرين 1.1: عدّ الكائنات باستخدام منطقة مضلعة

</div>


<div dir="rtl">

🛠️ شرح خطوات الكود البرمجي (خطوة بخطوة)

في هذا التمرين، سنقوم ببناء نظام متكامل لعدّ الكائنات العابرة لمنطقة محددة. ينقسم الكود إلى 5 مراحل أساسية:

#### **1. الإعدادات الأولية (Setup):**
*   نقوم بتعريف أسماء ملفات المخرجات وتحميل نموذج **YOLO** (استخدمنا النسخة `yolo26n` لسرعتها العالية).
*   نفتح الفيديو باستخدام `cv2.VideoCapture` لاستخراج أبعاده (العرض والارتفاع) وسرعته الأصلية (FPS).

#### **2. تحسين الأداء (Optimization):**
*   **لماذا `skip_rate`؟**
* معالجة كل إطار في الفيديو تستهلك وقتاً طويلاً. قمنا هنا بتقليل عدد الإطارات لتصبح **5 إطارات في الثانية** فقط. هذا يجعل البرنامج أسرع بـ 6 مرات تقريباً مع الحفاظ على دقة العدّ.

#### **3. تعريف "منطقة الرصد" (Counting Logic):**
*   نقوم برسم المنطقة (`lane_region`) التي نريد مراقبتها.
*   نستخدم أداة `solutions.ObjectCounter ؛ وهي أداة ذكية تقوم بربط نموذج YOLO بمنطقة العدّ، وتحديد الفئات المطلوبة (سيارات، شاحنات، إلخ).

#### **4. حلقة المعالجة (Processing Loop):**
*   نقرأ الفيديو إطاراً بإطار.
*   إذا كان الإطار متوافقاً مع معدل القفز الذي حددناه، نرسله للعداد (`counter`).
*   العداد يقوم برسم المربعات والخطوط والعدّ تلقائياً، ثم نحفظ هذا الإطار "المُحلل" في ملف فيديو جديد.


#### **5. معالجة المخرجات والعرض**

**لماذا نستخدم `ffmpeg`؟**  
بعض المتصفحات لا تدعم تشغيل فيديوهات OpenCV مباشرة، لذلك نستخدم FFmpeg لتحويل الفيديو إلى صيغة H.264 المتوافقة مع جميع المتصفحات وGoogle Colab.

**ما الذي يحدث في هذه المرحلة؟**
- تحويل الفيديو إلى صيغة متوافقة
- ضمان التشغيل بدون مشاكل
- تجهيز الفيديو للعرض

**النتيجة النهائية:**
- عرض الإحصائيات (عدد الداخلين والخارجين)
- تشغيل الفيديو بشكل واضح وسلس

---
**💡 نصيحة للطلاب:** يمكنك تعديل إحداثيات `lane_region` لتغيير مكان منطقة العدّ في الفيديو وتجربة النتائج بنفسك!

</div>


In [ ]:
import cv2
import os
from ultralytics import YOLO, solutions
from IPython.display import Video

# 1. إعداد أسماء الملفات وتحميل النموذج الذكي
temp_output = "temp_raw.mp4"
final_output = "lane_counting_5fps.mp4"
model = YOLO("weights/yolo26n.pt")

# 2. فتح الفيديو واستخلاص الأبعاد (العرض والارتفاع) والسرعة (FPS)
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 3. تحسين الأداء: تقليل عدد الإطارات (نستهدف 5 إطارات في الثانية) (نستهدف 5 إطارات في الثانية)
TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))
print(f"السرعة الأصلية: {original_fps} | معالجة إطار كل {skip_rate} إطارات")

# 4. رسم منطقة العد (خط وهمي أو مضلع)
lane_region = [(int(w*0.4), int(h*0.6)), (int(w*0.6), int(h*0.6))]

# تشغيل أداة العد الذكية (ObjectCounter)
counter = solutions.ObjectCounter(
    region=lane_region,
    model=model,
    classes=[2, 3, 5, 7], # فئات معينة: سيارات، شاحنات، إلخ
    show=False
)

# 5. تجهيز مسجل الفيديو لحفظ النتائج النهائية
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    # معالجة الإطارات المختارة فقط لسرعة التنفيذ
    if frame_idx % skip_rate == 0:
        results = counter(im0)
        annotated_frame = results.plot_im # استخراج الإطار بعد الرسم عليه
        writer.write(annotated_frame)     # تسجيل الإطار في ملف الفيديو

    frame_idx += 1
    if frame_idx > 300: break # التوقف بعد 300 إطار للتجربة السريعة

cap.release()
writer.release()

# 6. تحويل الفيديو لصيغة H.264 ليعمل على المتصفح باستخدام FFmpeg
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

print(f"📊 الإحصائيات النهائية: الداخل: {counter.in_count} | الخارج: {counter.out_count}")
Video(final_output, embed=True, width=800)


<div dir="rtl">

### 🏆 تحدي (اختياري)
جرب أشكالاً مختلفة للمناطق:
- خط أفقي عبر نصف الإطار
- مستطيل يغطي المسار الأيسر فقط
- أيها يعطيك نتائج أدق؟

</div>


<div dir="rtl">

---

## 🅿️ تمرين 1.2: إدارة مواقف السيارات

### 💡 الفكرة التعليمية:
تحديد أماكن المواقف مسبقاً وجعل YOLO يخبرنا فوراً عن الموقف الشاغر والمشغول.

**لماذا؟** لتسهيل حياة السائقين وتقليل الازدحام داخل المواقف.

**المهمة:** تحديد مواقف السيارات وتتبع حالة انشغالها مع مرور الوقت، وتسجيل النتائج في ملف CSV.

**الفيديو:** تصوير طائرة درون لمواقف سيارات أحد المولات.

</div>


In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# تحميل الملف إذا لم يكن موجوداً
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)


<div dir="rtl">

### 🏆 تحدي
أضف تنبيهاً فورياً عندما تتجاوز نسبة الإشغال 75%!

</div>


<div dir="rtl">

---

## 📋 تمرين 1.3: نظام التنبيه لانتظار الطوابير

### 💡 الفكرة التعليمية:
مراقبة منطقة الانتظار وتنبيه الإدارة إذا زاد الطابور عن حد معين.

**لماذا؟** لتحسين خدمة العملاء وتوزيع الموظفين بشكل أفضل.

**المهمة:** مراقبة منطقة معينة وإطلاق تنبيه عندما يتجاوز عدد الأشخاص في الطابور حداً معيناً.

</div>


In [ ]:
# 📋 تمرين 1.3: نظام التنبيه لانتظار الطوابير

import cv2
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO, solutions
from shapely.geometry import Point, Polygon
import numpy as np

# 1. الإعدادات
temp_output = "sahi_queue_raw.mp4"
final_output = "sahi_queue_5fps.mp4"
video_path = "Cars.mp4"

# تحميل النموذج باستخدام SAHI للكشف عن الأجسام الصغيرة
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="weights/yolo26n.pt", # استخدم الأوزان الخاصة بك
    confidence_threshold=0.3,
    device="cpu", # أو "cuda" إذا كان متاحاً
)

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. تحسين الأداء: تقليل عدد الإطارات لزيادة السرعة
TARGET_FPS = 3
skip_rate = max(1, int(original_fps / TARGET_FPS))

# 3. تحديد منطقة الانتظار (مضلع)
queue_points = [(int(w*0.3), int(h*0.5)), (int(w*0.7), int(h*0.5)),
                (int(w*0.9), int(h*0.9)), (int(w*0.1), int(h*0.9))]
queue_polygon = Polygon(queue_points)

# 4. حلقة المعالجة
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    # معالجة الإطارات المحددة فقط لتحقيق الهدف
    if frame_idx % skip_rate == 0:
        # إجراء استدلال مقسم (SAHI) للكشف عن الأجسام الصغيرة
        # هذا يحل مشكلة الكائنات الصغيرة في الفيديوهات عالية الدقة
        result = get_sliced_prediction(
            frame,
            detection_model,
            slice_height=640,
            slice_width=640,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2
        )

        occupancy_count = 0
        annotated_frame = frame.copy()

        # التعامل مع نتائج التوقعات
        for object_prediction in result.object_prediction_list:
            bbox = object_prediction.bbox.to_xyxy() # [x1, y1, x2, y2]
            cls_id = object_prediction.category.id

            # تصفية النتائج للسيارات والشاحنات فقط
            if cls_id in [2, 7]:
                # حساب نقطة المركز للكائن المكتشف
                cx, cy = int((bbox[0] + bbox[2]) / 2), int((bbox[1] + bbox[3]) / 2)

                # التحقق مما إذا كان الكائن داخل منطقة الانتظار
                if queue_polygon.contains(Point(cx, cy)):
                    occupancy_count += 1
                    color = (0, 255, 0) # لون أخضر إذا كان داخل الطابور
                else:
                    color = (255, 0, 0) # لون أزرق إذا كان خارجاً

                # رسم صندوق الإحاطة ونقطة المركز
                cv2.rectangle(annotated_frame, (int(bbox[0]), int(bbox[1])),
                              (int(bbox[2]), int(bbox[3])), color, 2)
                cv2.circle(annotated_frame, (cx, cy), 5, color, -1)

        # رسم مضلع منطقة الانتظار
        cv2.polylines(annotated_frame, [np.array(queue_points)], True, (255, 255, 0), 3)
        cv2.putText(annotated_frame, f"Queue Occupancy: {occupancy_count}", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

        writer.write(annotated_frame)

    frame_idx += 1
    if frame_idx > 300: break

cap.release()
writer.release()

# 5. الإنهاء وحفظ الفيديو
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

Video(final_output, embed=True, width=800)


<div dir="rtl">

---

# المحطة الثانية: القياسات الحركية والمكانية 🚀

سننتقل الآن من الموقع الثابت إلى دراسة "الحركة".

</div>


<div dir="rtl">

## 🚀 تمرين 2.1: تقدير السرعة

### 💡 الفكرة التعليمية:
حساب سرعة الأجسام بناءً على حركتها بين الإطارات.

**لماذا المعايرة (Calibration)؟** الكاميرا ترى بكسلات، ونحن نريد كم/ساعة. لذا نستخدم معامل `meter_per_pixel` للتحويل من عالم البكسل إلى عالم الأمتار.

**المهمة:** تقدير سرعة السيارات من خلال معايرة `meter_per_pixel`. حدد أسرع سيارة في الفيديو.

**تلميح:** ابدأ بـ `0.05` وقم بالتعديل بناءً على ما تراه مناسباً.

</div>


In [ ]:
# 🚀 تمرين 2.1: تقدير السرعة
video_path = "Cars.mp4"
temp_output = "speed_raw.mp4"
final_output = "speed_5fps.mp4"

# 1. تحميل النموذج البرمجي وفتح ملف الفيديو
model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. ضبط إعدادات السرعة والمعالجة (نقلل عدد الإطارات لزيادة الأداء)
TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

# 3. معايرة المسافات: تحويل البكسل إلى أمتار (تعديل هذا الرقم يحسن دقة حساب السرعة بالـ كم/س)
METER_PER_PIXEL = 0.05  # 🔧 جرب تغيير هذا الرقم بناءً على دقة النتائج

# 4. تشغيل أداة تقدير السرعة (SpeedEstimator) (SpeedEstimator)
speed_est = solutions.SpeedEstimator(
    show=False,
    model=model,
    fps=TARGET_FPS,
    meter_per_pixel=METER_PER_PIXEL,
    classes=[2, 3, 5, 7] # سيارات، حافلات، إلخ
)

# 5. حلقة المعالجة الرئيسية
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

speed_readings = []
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        results = speed_est(im0)
        writer.write(results.plot_im)

        # تخزين بيانات السرعة لعمل إحصائيات
        if hasattr(results, 'speed') and results.speed is not None:
            for track_id, speed in results.speed.items():
                if speed > 0:
                    speed_readings.append({'frame': frame_idx, 'track_id': track_id, 'speed_kmh': speed})

    frame_idx += 1
    if frame_idx > 200: break # معالجة أول 200 إطار فقط

cap.release()
writer.release()

# 6. التحويل النهائي والعرض
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

# عرض الإحصائيات (متوسط السرعة وأقصاها)
if speed_readings:
    df = pd.DataFrame(speed_readings)
    print(f"\n📊 متوسط السرعة: {df['speed_kmh'].mean():.1f} كم/س | أقصى سرعة: {df['speed_kmh'].max():.1f} كم/س")
else:
    print("⚠️ لم يتم التقاط بيانات للسرعة")

Video(final_output, embed=True, width=800)


<div dir="rtl">

### 🏆 تحدي
قم بتعديل معامل `meter_per_pixel` للحصول على سرعات أكثر واقعية. ما هي القيمة التي وجدتها الأنسب؟

</div>


<div dir="rtl">

---

## 📏 تمرين 2.2: حساب المسافات

### 💡 الفكرة التعليمية:
حساب المسافة الفعلية بين كائنين.

**لماذا؟** ضروري جداً في أنظمة القيادة الذاتية لمنع التصادم.

**المهمة:** حساب المسافة بين سيارتين يتم تتبعهما. أطلق تحذيراً إذا كانت المسافة قريبة جداً.

</div>


In [ ]:
# 📏 تمرين 2.2: حساب المسافة

video_path = "Cars.mp4"
temp_output = "distance_raw.mp4"
final_output = "distance_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

dist_calc = solutions.DistanceCalculation(
    show=False,
    model=model,
    classes=[2, 3, 5, 7]
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        results = dist_calc(im0)
        writer.write(results.plot_im)

    frame_idx += 1
    if frame_idx > 150: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)


<div dir="rtl">

---

# المحطة الثالثة: الخصوصية والتمثيل البصري 🔒

كيف نحول البيانات إلى صور ورسوم يفهمها الجميع؟

</div>


<div dir="rtl">

## 🔒 تمرين 3.1: إخفاء الهوية (التغبيش)

### 💡 الفكرة التعليمية:
طمس ملامح الأشخاص تلقائياً.

**لماذا؟** للامتثال لقوانين حماية البيانات والخصوصية في الأماكن العامة.

**المهمة:** تغبيش الأشخاص المكتشفين للامتثال لمعايير الخصوصية.

**تحدي:** حاول تغبيش الوجوه فقط باستخدام نقاط وضعية الجسم (متقدم).

</div>


In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# تحميل الملف إذا لم يكن موجوداً
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)


<div dir="rtl">

### 🏆 تحدي متقدم: تغبيش الوجه فقط
استخدم نموذج YOLO Pose (`yolov8n-pose.pt`) وقم بتغبيش نقاط الوجه فقط (النقاط من 0 إلى 4).

</div>


<div dir="rtl">

---

## 🔥 تمرين 3.3: الخرائط الحرارية (Heatmap)

### 💡 الفكرة التعليمية:
تلوين المناطق حسب كثافة الحركة فيها.

**لماذا؟** لمعرفة "النقاط الساخنة" التي يتجمع فيها الناس أو السيارات بكثرة.

**المهمة:** توليد خريطة حرارية توضح كثافة الحركة، وقارن بين خرائط الألوان المختلفة.

</div>


In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# تحميل الملف إذا لم يكن موجوداً
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)


<div dir="rtl">

---

# المحطة الرابعة: تطبيقات متخصصة 🏋️

تطبيقات تتجاوز مجرد كشف الصناديق.

</div>


<div dir="rtl">

## 🏋️ تمرين 4.1: مراقبة التمارين الرياضية

### 💡 الفكرة التعليمية:
استخدام نقاط مفاصل الجسم لعدّ التكرارات الرياضية.

**لماذا؟** لتحويل الكاميرا إلى مدرب شخصي يراقب دقتك في أداء التمرين.

**المهمة:** عدّ تمارين الضغط باستخدام YOLO Pose، وقم بتعديل الكود ليناسب تمارين أخرى.

**الفيديو:** Person doing pull-ups

</div>


In [ ]:
# 🏋️ تمرين 4.1: مراقبة التمارين الرياضية

video_path = "Pull_ups.mp4"
temp_output = "gym_raw.mp4"
final_output = "gym_5fps.mp4"

model = YOLO("yolov8n-pose.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

gym = solutions.AIGym(
    show=False,
    kpts=[6, 8, 10],  # كتف - مرفق - معصم
    model=model
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    if frame_idx % skip_rate == 0:
        results = gym(im0)
        writer.write(results.plot_im)

    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

print(f"💪 عدد التكرارات: {getattr(gym, 'count', 'N/A')}")

from IPython.display import Video
Video(final_output, embed=True, width=800)


<div dir="rtl">

### 🏆 تحدي
قم بتعديل نقاط المفاصل لتناسب تمارين أخرى:
- **تمرين القرفصاء (Squats):** استخدم النقاط [11, 13, 15] (ورك-ركبة-كاحل).
- **تمرين القفز (Jumping jacks):** استخدم النقاط [5, 7, 9] + [6, 8, 10] (كلا الذراعين).

</div>


<div dir="rtl">

---

## 🎭 تمرين 4.2: تتبع الأشكال الدقيقة (Segmentation)

### 💡 الفكرة التعليمية:
تتبع الكائنات بحدودها الدقيقة (بكسل ببكسل) وليس مجرد مربعات.

**لماذا؟** للدقة العالية في فصل الكائنات عن الخلفية.

**المهمة:** تتبع الكائنات باستخدام أقنعة دقيقة (Segmentation) وقارنها بالتتبع العادي.

</div>


In [ ]:
# 🎭 تمرين 4.2: تتبع الأشكال الدقيقة (Segmentation)

video_path = "Cars.mp4"

# استخدام نموذج YOLO الخاص بالتقسيم (Segmentation)
model = YOLO("yolo26n-seg.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

frame_count = 0
max_إطارات = 50

# التتبع باستخدام نموذج التقسيم (Segmentation)
results = model.track(source=video_path, stream=True, classes=[2, 3, 5, 7], persist=True)

tracked_objects = set()

for result in results:
    if result.boxes.id is not None:
        for track_id in result.boxes.id.cpu().numpy():
            tracked_objects.add(int(track_id))
    frame_count += 1
    if frame_count >= max_إطارات:
        break

print(f"✓ تم تتبع {len(tracked_objects)} مركبات فريدة باستخدام أقنعة التقسيم")


<div dir="rtl">

---

# المشروع الختامي: نظام ذكاء الأعمال للمتاجر 🎯

### 🎯 التحدي الجماعي (Capstone Project):
قم ببناء نظام متكامل يجمع 3 حلول على الأقل لتحليل فيديو متجر (عد زبائن، خريطة حرارية، تغبيش وجوه).

## 🎯 تحدي المجموعة (3-4 طلاب)

قم ببناء نظام متكامل لتحليل بيانات التجزئة يقوم بما يلي:

1. **عد الزبائن** الداخلين للمتجر (ObjectCounter)
2. **إنشاء خريطة حرارية** للممرات الأكثر طلباً (Heatmap)
3. **تغبيش الوجوه** لضمان الخصوصية (ObjectBlurrer)
4. **عرض ساعات الذروة** باستخدام الرسوم البيانية (Analytics)
5. **تنبيه** إذا كان طابور الكاشير > 3 أشخاص (Queue alert)

### المتطلبات
- استخدام **3 حلول مختلفة على الأقل**
- معالجة الفيديو بالكامل من البداية للنهاية
- تصدير النتائج إلى ملف CSV/JSON

### أهداف إضافية
- إضافة تقدير السرعة لتحليل أنماط مشي الزبائن
- إنشاء لوحة بيانات بسيطة للعرض
- دمج فيديوهات متعددة

### المخرجات
- مفكرة (Notebook) تعمل وتحتوي على جميع الحلول المدمجة
- ملف `retail_summary.csv` يحتوي على الأعداد، أوقات الذروة، ونسبة الإشغال
- عرض تجريبي لمدة دقيقتين لكل مجموعة

</div>


In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# تحميل الملف إذا لم يكن موجوداً
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)


<div dir="rtl">

---

## ✅ أسئلة ختامية ومراجعة

1. أي من الحلول (Solutions) كانت الأكثر تحدياً بالنسبة لك أثناء الإعداد؟ ولماذا؟
2. كيف يمكنك تطويع هذه الحلول لاستخدامها في مشروع حقيقي على أرض الواقع (Real Deployment)؟
3. ما هي المشاكل التجارية أو الصناعية الأخرى التي تعتقد أن هذه الحلول يمكن أن تعالجها؟

---

## 📚 المصادر والمراجع

- [Object Counting Docs](https://docs.ultralytics.com/guides/object-counting/)
- [Heatmaps Docs](https://docs.ultralytics.com/guides/heatmaps/)
- [Speed Estimation Docs](https://docs.ultralytics.com/guides/speed-estimation/)
- [Analytics Docs](https://docs.ultralytics.com/guides/analytics/)
- [Parking Management Docs](https://docs.ultralytics.com/guides/parking-management/)
- [Workouts Monitoring Docs](https://docs.ultralytics.com/guides/workouts-monitoring/)

---

**نهاية المعمل - أحسنت!**

</div>
